In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import os

from specula.mmlib.yaml_overrides import write_yaml_overrides
from specula.mmlib.compute_rec import compute_and_save_rec
from specula.mmlib.utils import get_pupil_mask

from utils import calib_dir, result_dir, run_specula, rec_mmse

In [2]:
import matplotlib as mpl
from cycler import cycler

cmap = mpl.colormaps['hot'] # 1. Choose your colormap (e.g., 'hot', 'jet', 'turbo')
colors = cmap(np.linspace(0.0, 1.0, 7)) # 2. Extract a set of colors from it (e.g., 6 distinct colors across the map)
# plt.rcParams['axes.prop_cycle'] = cycler(color=colors) # 3. Apply it to the default line cycle
plt.rcParams['image.cmap'] = 'magma' # 4. Change the default colormap for images/scatter plots
plt.rcParams.update({
    "text.usetex": True, # Use LaTeX to write all text
    "font.family": "serif", # Use a serif font
    "font.serif": ["Computer Modern Roman"] # Specifically use the classic LaTeX font
})
plt.rcParams['image.origin'] = 'lower'

In [ ]:
nSubaps = [10,14,24,36,48]
rMods = [0,0.5,1,2,3]
# nModes = [54,100,300,800,1200,1600]
nModes = [54,100,300,800,1300]
wvls = [900,1300]
overrides_yaml = 'temp_overrides'

# calib_dir, result_dir: imported from utils

In [ ]:
# Generate pupil data
for nSubap in nSubaps:
    tag = f'pyr_pupdata_{nSubap}x{nSubap}'
    try:
        pupdata = fits.getdata(os.path.join(calib_dir,'pupils',tag+'.fits'))
    except FileNotFoundError:
        ovdes = ("{"
            f"pyr_vis.pup_diam: {nSubap}, "
            f"pyr_vis.pup_dist: {np.max((14,np.ceil(nSubap*60/48)))}, "
            f"pyr_pupdata.output_tag: {tag}, "
            "}")
        write_yaml_overrides(ovdes,temp_name=overrides_yaml)
        run_specula('ristretto_main.yml', 'calib_pupdata.yml', overrides_yaml, log_file='specula.log')

In [ ]:
# Generate IM data
for wvl in wvls:
    for rMod in rMods:
        for nSubap in nSubaps:
            tag = f'pyr{rMod:1.1f}_{nSubap:1.0f}x{nSubap:1.0f}_wvl{wvl:1.0f}_bmc2k_vlt'
            im_tag = tag+'_im'
            try:
                im = fits.getdata(os.path.join(calib_dir,'im',im_tag+'.fits'),1)
            except FileNotFoundError:
                print(f'Simulating: {nSubap}x{nSubap}, {wvl}nm, {rMod:1.1f} '+'$\lambda$/D')
                ovdes = ("{"
                    f"pyr_vis.pup_diam: {nSubap}, "
                    f"pyr_vis.pup_dist: {np.max((14,np.ceil(nSubap*60/48)))}, "
                    f"source_vis.wavelengthInNm: {wvl}, "
                    f"pyr_vis.mod_amp: {rMod:1.1f}, "
                    f"pyr_im_calibrator.im_tag: {im_tag}, "
                    f"pyr_vis.wavelengthInNm: {wvl}, "
                    f"pyr_slopes_vis.pupdata_object: pyr_pupdata_{nSubap}x{nSubap}, "
                    "}")
                write_yaml_overrides(ovdes,temp_name=overrides_yaml)
                run_specula('ristretto_main.yml', 'calib_im.yml', overrides_yaml, log_file='specula.log')

                for N in nModes:
                    rec_tag = tag+f'_{N:1.0f}modes_rec'
                    compute_and_save_rec(root_dir=calib_dir, im_tag=im_tag, rec_tag=rec_tag, Nmodes=N)

In [ ]:
mask = get_pupil_mask(npix=120,filepath=os.path.join(calib_dir,'pupils','pyr_pupdata_48x48.fits'))
os.makedirs(os.path.join(calib_dir,'aliasing'),exist_ok=True)

dt = 5e-4
for rMod in rMods:
    for j,nSubap in enumerate(nSubaps):
        for i,N in enumerate(nModes):
            if i <= j:
                tag = f'pyr{rMod:1.1f}_{nSubap:1.0f}x{nSubap:1.0f}'
                alias_vis_tag = tag+f'_{N}corrmodes_{N}modes_wvl900_{1/dt:1.0f}Hz_alias'
                alias_nir_tag = tag+f'_{N}corrmodes_{N}modes_wvl1300_{1/dt:1.0f}Hz_alias'
                frame_vis_tag = tag+f'_{N}corrmodes_wvl900_frame'
                frame_nir_tag = tag+f'_{N}corrmodes_wvl1300_frame'
                try:
                    alias_vis = fits.getdata(os.path.join(calib_dir,'aliasing',alias_vis_tag+'.fits'))
                    frames_vis = fits.getdata(os.path.join(calib_dir,'aliasing',frame_vis_tag+'.fits'))
                    alias_nir = fits.getdata(os.path.join(calib_dir,'aliasing',alias_nir_tag+'.fits'))
                    frames_nir = fits.getdata(os.path.join(calib_dir,'aliasing',frame_nir_tag+'.fits'))
                except FileNotFoundError:
                    print(f'Simulating: {nSubap}x{nSubap}, {N} corrected modes, 900+1300nm, {rMod:1.1f} '+'$\lambda$/D')
                    ovdes = ("{"
                        f"main.time_step: {dt:1.5f}, "
                        f"atmo_modes.nmodes: {N}, "
                        f"pyr_vis.pup_diam: {nSubap}, "
                        f"pyr_vis.pup_dist: {np.max((14,np.ceil(nSubap*60/48)))}, "
                        f"pyr_vis.mod_amp: {rMod:1.1f}, "
                        f"cred_vis.dt: {dt:1.5f}, "
                        f"pyr_slopes_vis.pupdata_object: pyr_pupdata_{nSubap}x{nSubap}, "
                        f"pyr_nir.pup_diam: {nSubap}, "
                        f"pyr_nir.pup_dist: {np.max((14,np.ceil(nSubap*60/48)))}, "
                        f"pyr_nir.mod_amp: {rMod:1.1f}, "
                        f"cred_nir.dt: {dt:1.5f}, "
                        f"pyr_slopes_nir.pupdata_object: pyr_pupdata_{nSubap}x{nSubap}, "
                        "}")
                    write_yaml_overrides(ovdes,temp_name=overrides_yaml)
                    run_specula('ristretto_main.yml', 'calib_wfs_aliasing.yml', overrides_yaml, log_file='specula.log')

                    # Save average frame
                    frames_vis = np.mean(fits.getdata(os.path.join(result_dir,'vis_frames.fits')),axis=0)
                    frames_nir = np.mean(fits.getdata(os.path.join(result_dir,'nir_frames.fits')),axis=0)
                    fits.writeto(os.path.join(calib_dir,'aliasing',frame_vis_tag+'.fits'),frames_vis,overwrite=True)
                    fits.writeto(os.path.join(calib_dir,'aliasing',frame_nir_tag+'.fits'),frames_nir,overwrite=True)

                    # Get slopes
                    slopes_vis = fits.getdata(os.path.join(result_dir,'slopes_vis.fits'))
                    slopes_nir = fits.getdata(os.path.join(result_dir,'slopes_nir.fits'))
                    for Nm in nModes:
                        if Nm <= N:
                            alias_vis_tag = tag+f'_{N}corrmodes_{Nm}modes_wvl900_{1/dt:1.0f}Hz_alias'
                            alias_nir_tag = tag+f'_{N}corrmodes_{Nm}modes_wvl1300_{1/dt:1.0f}Hz_alias'
                            rec_vis = fits.getdata(os.path.join(calib_dir,'rec',f"pyr{rMod:1.1f}_{nSubap:1.0f}x{nSubap:1.0f}_wvl900_bmc2k_vlt_{Nm}modes_rec.fits"))
                            rec_nir = fits.getdata(os.path.join(calib_dir,'rec',f"pyr{rMod:1.1f}_{nSubap:1.0f}x{nSubap:1.0f}_wvl1300_bmc2k_vlt_{Nm}modes_rec.fits"))
                            modes_vis = (rec_vis @ slopes_vis.T).T
                            modes_nir = (rec_nir @ slopes_nir.T).T
                            alias_vis = np.var(modes_vis,axis=0)
                            alias_nir = np.var(modes_nir,axis=0)
                            fits.writeto(os.path.join(calib_dir,'aliasing',alias_vis_tag+'.fits'),alias_vis)
                            fits.writeto(os.path.join(calib_dir,'aliasing',alias_nir_tag+'.fits'),alias_nir)

In [73]:
# Compute MMSE rec for unmodulated pyramid
rMod = 0

for j,nSubap in enumerate(nSubaps):
    for i,N in enumerate(nModes):
        if i <= j:
            im_vis = fits.getdata(os.path.join(calib_dir,'im',f"pyr{rMod:1.1f}_{nSubap:1.0f}x{nSubap:1.0f}_wvl900_bmc2k_vlt_im.fits"))
            im_nir = fits.getdata(os.path.join(calib_dir,'im',f"pyr{rMod:1.1f}_{nSubap:1.0f}x{nSubap:1.0f}_wvl1300_bmc2k_vlt_im.fits"))
            pup = fits.getdata(os.path.join(calib_dir,'pupils',f"pyr_pupdata_{nSubap:1.0f}x{nSubap:1.0f}.fits"),1)
            tag = f'pyr{rMod:1.1f}_{nSubap:1.0f}x{nSubap:1.0f}'
            frame_vis_tag = tag+f'_{N}corrmodes_wvl900_frame'
            frame_nir_tag = tag+f'_{N}corrmodes_wvl1300_frame'
            frame_vis = fits.getdata(os.path.join(calib_dir,'aliasing',frame_vis_tag+'.fits'))
            frame_nir = fits.getdata(os.path.join(calib_dir,'aliasing',frame_nir_tag+'.fits'))
            rec_vis,_ = rec_mmse(im_vis[:,:N],frame_vis,pup,ron_var=0.5**2,use_cross=True)
            rec_nir,_ = rec_mmse(im_nir[:,:N],frame_nir,pup,ron_var=0.5**2,use_cross=True)
            fits.writeto(os.path.join(calib_dir,'rec',f'pyr{rMod:1.1f}_{nSubap:1.0f}x{nSubap:1.0f}_wvl900_bmc2k_vlt_{N}modes_mmse_rec.fits'),rec_vis,overwrite=True)
            fits.writeto(os.path.join(calib_dir,'rec',f'pyr{rMod:1.1f}_{nSubap:1.0f}x{nSubap:1.0f}_wvl1300_bmc2k_vlt_{N}modes_mmse_rec.fits'),rec_nir,overwrite=True)
            

In [7]:
# slopes_vis = fits.getdata(os.path.join(result_dir,'slopes_vis.fits'))
# slopes_nir = fits.getdata(os.path.join(result_dir,'slopes_nir.fits'))

# print(slopes_vis.shape,slopes_nir.shape)
# rec_vis = fits.getdata(os.path.join(calib_dir,'rec',f"pyr{rMod:1.1f}_{nSubap:1.0f}x{nSubap:1.0f}_wvl900_bmc2k_vlt_{Nm}modes_rec.fits"))
# rec_nir = fits.getdata(os.path.join(calib_dir,'rec',f"pyr{rMod:1.1f}_{nSubap:1.0f}x{nSubap:1.0f}_wvl1300_bmc2k_vlt_{Nm}modes_rec.fits"))
# modes_vis = (rec_vis @ slopes_vis.T).T
# modes_nir = (rec_nir @ slopes_nir.T).T
# alias_vis = np.var(modes_vis,axis=0)
# alias_nir = np.var(modes_nir,axis=0)
